<a href="https://colab.research.google.com/github/Joydeb123-Bharat/MIPS32-Processor/blob/main/SimpleRISC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

SimpleRISC Assembler


In [ ]:
isa_opcodes = {
    "ADD":  "00000", "SUB":  "00001", "MUL":  "00010", "DIV":  "00011",
    "MOD":  "00100", "CMP":  "00101", "AND":  "00110", "OR":   "00111",
    "NOT":  "01000", "MOV":  "01001", "LSL":  "01010", "LSR":  "01011",
    "ASR":  "01100", "NOP":  "01101", "LD":   "01110", "ST":   "01111",
    "BEQ":  "10000", "BGT":  "10001", "B":    "10010", "CALL": "10011",
    "RET":  "10100", "XOR":  "10101", "HLT":  "10110"
}

registers = {
    "R0":  "0000", "R1":  "0001", "R2":  "0010", "R3":  "0011",
    "R4":  "0100", "R5":  "0101", "R6":  "0110", "R7":  "0111",
    "R8":  "1000", "R9":  "1001", "R10": "1010", "R11": "1011",
    "R12": "1100", "R13": "1101", "R14": "1110", "R15": "1111"
}
# Removing the Comment lines
def read_from_file(filename):
    lines = []
    with open(filename, 'r') as file:
        for line in file:
            no_comments = line.split('//')[0]
            clean_line = no_comments.strip().upper()
            if clean_line:
                lines.append(clean_line)
    return lines
#Extracting the opcode, registers and immediates
def parse_line(line):
    clean_line = line.replace(',', ' ').replace('[', ' ').replace(']', ' ')
    parts = clean_line.split()
    return parts
#Constructing the Symbol table
def first_pass(lines):
    symbol_table = {}
    clean_instructions = []
    pc = 0
    for line in lines:
        if ':' in line:
            parts = line.split(':')
            label_name = parts[0].strip()
            symbol_table[label_name] = pc
            instruction = parts[1].strip()
        else:
            instruction = line.strip()

        if instruction:
            clean_instructions.append(instruction)
            pc += 4
    return symbol_table, clean_instructions

#For Decimal to Binary
def dec_to_bin(num_str, bits):
    num = int(num_str)
    if num >= 0:
        return format(num, f'0{bits}b')
    else:
        return format((1 << bits) + num, f'0{bits}b')
#Instruction Encodings
def ass_0_type(parts):
    opcode = isa_opcodes[parts[0]]
    padding = "0" * 27
    return opcode + padding

def ass_1_type(parts, current_pc, symbol_table):
    if parts[1] in symbol_table:
        offset_int = int((symbol_table[parts[1]] - current_pc) / 4)
    else:
        offset_int = int(parts[1])
    offset_bin = dec_to_bin(str(offset_int), 27)
    opcode = isa_opcodes[parts[0]]
    return opcode + offset_bin

def ass_2_type_mov_not(parts, modifier):
    opcode = isa_opcodes[parts[0]]
    rd = registers[parts[1]]
    empty_rs1 = "0000"
    if parts[2] in registers:
        r_i = "0"
        rs2 = registers[parts[2]]
        padding = "0" * 14
        return opcode + r_i + rd + empty_rs1 + rs2 + padding
    else:
        r_i = "1"
        mod = "00"
        if modifier == 'U':
            mod = "01"
        elif modifier == 'H':
            mod = "10"
        imm = dec_to_bin(parts[2], 16)
        return opcode + r_i + rd + empty_rs1 + mod + imm

def ass_2_type_cmp(parts):
    opcode = isa_opcodes[parts[0]]
    rd = "0000"
    rs1 = registers[parts[1]]
    if parts[2] in registers:
        r_i = "0"
        rs2 = registers[parts[2]]
        padding = "0" * 14
        return opcode + r_i + rd + rs1 + rs2 + padding
    else:
        r_i = "1"
        imm = dec_to_bin(parts[2], 18)
        return opcode + r_i + rd + rs1 + imm

def ass_r_type(parts):
    opcode = isa_opcodes[parts[0]]
    r = "0"
    rd = registers[parts[1]]
    rs1 = registers[parts[2]]
    rs2 = registers[parts[3]]
    padding = "0" * 14
    return opcode + r + rd + rs1 + rs2 + padding

def ass_i_type(parts, modifier):
    opcode = isa_opcodes[parts[0]]
    i = "1"
    rd = registers[parts[1]]
    rs1 = registers[parts[2]]
    mod = "00"
    if modifier == 'U':
        mod = "01"
    elif modifier == 'H':
        mod = "10"
    imm = dec_to_bin(parts[3], 16)
    return opcode + i + rd + rs1 + mod + imm

def ass_mem_type(parts, modifier):
    opcode = isa_opcodes[parts[0]]
    r_i = "1"
    rd = registers[parts[1]]
    mod = "00"
    if modifier == 'U':
        mod = "01"
    elif modifier == 'H':
        mod = "10"
    imm = dec_to_bin(parts[2], 16)
    rs1 = registers[parts[3]]
    return opcode + r_i + rd + rs1 + mod + imm

def assemble_instruction(parts, current_pc, symbol_table):
    modifier = "N"

    if parts[0][-1] in ["U", "H"] and parts[0][:-1] in isa_opcodes:
        modifier = parts[0][-1]
        parts[0] = parts[0][:-1]

    if len(parts) == 1:
        return ass_0_type(parts)

    elif len(parts) == 2:
        return ass_1_type(parts, current_pc, symbol_table)

    elif len(parts) == 3:
        if parts[0] in ["MOV", "NOT"]:
            return ass_2_type_mov_not(parts, modifier)
        elif parts[0] == "CMP":
            return ass_2_type_cmp(parts)

    elif len(parts) == 4:
        if parts[0] in ["LD", "ST"]:
            return ass_mem_type(parts, modifier)
        elif parts[3] in registers:
            return ass_r_type(parts)
        else:
            return ass_i_type(parts, modifier)

    return "Invalid Instruction"

def run_assembler(filename):
    raw_lines = read_from_file(filename)
    symbol_table, clean_instructions = first_pass(raw_lines)

    machine_code = []
    current_pc = 0

    for instruction in clean_instructions:
        parts = parse_line(instruction)
        binary32 = assemble_instruction(parts, current_pc, symbol_table)

        if binary32 == "Invalid Instruction":
            print(f"Invalid instruction: {instruction}")
            return None

        machine_code.append(binary32)
        current_pc += 4

    return machine_code

final_binary = run_assembler(input("Enter the name of the file: "))
if final_binary:
    for line in final_binary:
        print(line)

Enter the name of the file: Comprehensive SimpleRISC Test Prog.txt
01001100000000000000000000000000
01001100010000010000000001100100
01001100100000100000000011111111
00000000110001001000000000000000
00001001000011000100000000000000
00010001010001000100000000000000
10101001100101010000000000000000
01111101100000000000000000000100
01110101110000000000000000000100
00001100010001000000000000000001
00101000000001000000000000000000
10001111111111111111111111111110
10011000000000000000000000000010
10010000000000000000000000000011
01000001110000011100000000000000
10100000000000000000000000000000
01101000000000000000000000000000
10110000000000000000000000000000
